# Experiment 4.0.3 — Hierarchical Fixed250 Multi-Spike SNN

Analysis only. Training is performed by the Slurm jobs under `scripts/bash_script/SNN_Bash/`. All models receive exactly the same scaled Fixed250 vectors; no sub-bin timing is reconstructed.

Primary question: does `Local128 -> RSNN128` outperform both the reused 1-layer RSNN128 Multi-HO baseline and the nearly parameter-matched RSNN176 control? The endpoint-state Linear probe is diagnostic only and sees one causal endpoint membrane vector, not temporal phases.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
root = repo_root / 'notebooks' / 'artifacts' / 'experiment_4_0_3_hierarchical_snn' / 'fixed250_hierarchical_multispike_v1'
runs_path = root / 'runs.csv'
summary_path = root / 'summary.csv'
effects_path = root / 'paired_effects.csv'
refs_path = root / 'references.csv'

if not runs_path.exists():
    raise FileNotFoundError(f'Finalized Exp4.0.3 artifacts not found: {runs_path}')

runs = pd.read_csv(runs_path)
summary = pd.read_csv(summary_path)
effects = pd.read_csv(effects_path)
refs = pd.read_csv(refs_path)
summary

In [ ]:
test = runs[runs['split'] == 'test'].copy()
order = [
    'rsnn128_multispike_reused',
    'rsnn176_capacity',
    'local128_ff128',
    'rsnn128_rsnn128',
    'local128_rsnn128',
]
table = (
    test.groupby('condition')[['valid_count_balanced_accuracy', 'state_probe_balanced_accuracy']]
    .agg(['mean', 'std'])
    .reindex(order)
)
table

In [ ]:
plot_data = test.groupby('condition')['valid_count_balanced_accuracy'].agg(['mean', 'std']).reindex(order)
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(plot_data)), plot_data['mean'], yerr=plot_data['std'], capsize=4)
ax.set_xticks(range(len(plot_data)))
ax.set_xticklabels(plot_data.index, rotation=25, ha='right')
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Fully-spiking Fixed250 decoder')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
probe_data = test.groupby('condition')['state_probe_balanced_accuracy'].agg(['mean', 'std']).reindex(order)
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(probe_data)), probe_data['mean'], yerr=probe_data['std'], capsize=4)
ax.set_xticks(range(len(probe_data)))
ax.set_xticklabels(probe_data.index, rotation=25, ha='right')
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Post-hoc endpoint-state Linear probe')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
display(effects.groupby('condition')[['fully_spiking_delta_vs_baseline', 'state_probe_delta_vs_baseline']].agg(['mean', 'std']))
display(refs)